# SpiderNet data loading and MI-dimension-selection tutorial (Pancancer)

This notebook is a step-by-step tutorial for loading and preprocessing the Pancancer spatial transcriptomics data for SpiderNet, as well as running the MI-dimension-selection procedure on the processed data.

1. **Basic user inputs**
2. **Pancancer-specific SpiderNet input assumptions**
3. **Build the base config**
4. **Preview the LR-correlation density before choosing `lr_corr_threshold`**
5. **Run the full unified loader**
6. **Inspect outputs**
7. **Save Pancancer-specific batch metadata**
8. **Run MI dimension selection on the processed outputs**

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

# If the notebook is run from the project root, this is usually enough.
sys.path.append(str(Path.cwd()))

from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import get_default_cellchat_db, get_default_scseqcomm_db

C:\Users\junji\miniconda3\envs\SpiderNet_env\Lib\site-packages\louvain\__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Step 1. Basic user inputs

Edit the variables in this section first.

In [2]:

DATA_ROOT = Path("D:/SpiderNet/Data/Pancancer")  ## Set this to your local Pancancer data directory
# For entire pan-cancer data
OUTPUT_DIR = Path("D:/SpiderNet/Results/Pancancer/ProcessedData_entire")  ## Set this to your desired processed-data directory for Pancancer
# ## For a subset of pan-cancer data
# OUTPUT_DIR = Path("D:/SpiderNet/Results/Pancancer/ProcessedData")  ## Set this to your desired processed-data directory for Pancancer

SPECIES = "human"  ## Species used for ligand-receptor database loading. Options are "human" or "mouse".

SAMPLE_COL = "SampleID"  ## Pancancer batches are defined as <file_prefix>_subslice<subslice_id>, matching the original Pancancer loader.
CELL_CLASS_COL = "celltype_final"  ## The cell type annotation column in adata.obs.
SPATIAL_KEY = "spatial"  ## Spatial coordinates are read from adata.obsm[SPATIAL_KEY].
PYG_EXTRA_OBS_FIELDS = {}  ## Extra adata.obs fields to copy into each PyG sample object. Leave empty unless you want to propagate additional metadata.

N_HVG = 1000  ## Number of top highly variable genes used for SpiderNet model training.
N_HVG_LR = 2000  ## Number of top highly variable genes used when selecting ligand-receptor pairs from the training gene space.
NUM_NEIGHBORS = 5  ## Number of spatial neighbors. The Pancancer_modeltraining notebook uses 5 for this large dataset.

LR_CORR_THRESHOLD = None  ## Set this to the desired ligand-receptor correlation threshold after inspecting the preview plot in Step 4. In the original PerturbFISH loader, the final LR-correlation filter uses 0.5.

LR_LIST_PATH = None  ## Optional path to a predefined LR list (.pkl/.npy/.txt/.csv). If provided, the original Pancancer loader skips LR activation/correlation filtering.
GENE_LIST_PATH = None  ## Optional path to a predefined gene list (.pkl/.npy/.txt/.csv). If provided, the original Pancancer loader skips HVG selection.
# For entire pan-cancer data
ADATA_FOLDER_NAME = "adata_entire"  ## Folder under DATA_ROOT that stores the .h5ad files.
# ## For a subset of pan-cancer data
# ADATA_FOLDER_NAME = "adata"  ## Folder under DATA_ROOT that stores the .h5ad files.

## Step 2. Pancancer-specific SpiderNet input

This notebook follows the Pancancer data convention used in the original
`dataloading_Pancancer` script and the `Pancancer_modeltraining` notebook:

- `.h5ad` files are read from `DATA_ROOT / ADATA_FOLDER_NAME`
- expression is read directly from `X`
- zero-count cells are removed
- if `X` looks like raw integer counts, the loader runs `normalize_total(..., target_sum=1e4)` and `log1p`
- cell names are prefixed with the sample prefix extracted from the filename
- each cell is assigned a batch label  
  `SampleID = <file_prefix>_subslice<subslice_id>`
- spatial coordinates are stored in `obsm[SPATIAL_KEY]`
- cell types are defined by `obs["celltype_final"]`
- the Pancancer loader supports optional predefined LR lists and predefined gene lists
- the original Pancancer loader uses `lr_corr_threshold = 0.2` when LR filtering is enabled

In [3]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)

def normalize_optional_path_local(path_str):
    if path_str is None:
        return None
    path_str = str(path_str).strip()
    if path_str.lower() in {"", "none", "null", "nan"}:
        return None
    return path_str

def pancancer_file_prefix(context):
    return Path(context["file_name"]).stem.split("_")[0]

def pancancer_per_file_hook(adata, context):
    sample_prefix = pancancer_file_prefix(context)

    if sp.issparse(adata.X):
        adata.X = adata.X.astype(np.float32)
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)

    adata.obs_names = [f"{sample_prefix}_{cellname}" for cellname in adata.obs_names]

    if "subslice_id" not in adata.obs.columns:
        raise KeyError(
            "Pancancer preprocessing expects adata.obs['subslice_id'] so that "
            "SampleID can be constructed as <file_prefix>_subslice<subslice_id>."
        )

    adata.obs["source_file_prefix"] = sample_prefix
    adata.obs["SampleID"] = [
        f"{sample_prefix}_subslice{subslice_id}"
        for subslice_id in adata.obs["subslice_id"].astype(str)
    ]

    if SPATIAL_KEY not in adata.obsm:
        raise KeyError(
            f"Pancancer preprocessing expects adata.obsm[{SPATIAL_KEY!r}] to contain spatial coordinates."
        )

    adata.layers.clear()
    for key in ["X_pca", "X_umap"]:
        if key in adata.obsm:
            del adata.obsm[key]
    adata.obsm = {"spatial": np.asarray(adata.obsm[SPATIAL_KEY])}
    adata.varm.clear()
    adata.raw = None
    if adata.var.shape[1] > 0:
        adata.var = adata.var.iloc[:, []].copy()

    return adata

APPLY_HVG_SELECTION = normalize_optional_path_local(GENE_LIST_PATH) is None
APPLY_LR_CORR_FILTER = normalize_optional_path_local(LR_LIST_PATH) is None

DATA_REPRESENTATION_CONFIG = {
    "expression_source": {"kind": "X", "name": None},
    "normalize_strategy": "auto",
    "log1p": True,
    "remove_zero_count_cells": True,
    "spatial_source": {"kind": "obsm", "key": "spatial"},
    "apply_hvg_selection": APPLY_HVG_SELECTION,
}

## Step 3. Build the base config

This config contains the common loading settings for the Pancancer dataset.
`lr_corr_threshold` is intentionally injected at run time only when LR correlation
filtering is enabled.

In [4]:
base_config = {
    "data_path_main": DATA_ROOT,
    "output_dir": OUTPUT_DIR,
    "adata_folder_name": ADATA_FOLDER_NAME,
    "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
    "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,

    "sample_col": SAMPLE_COL,
    "sample_name_obs_col": "source_file_prefix",
    "sample_attr_obs_col": SAMPLE_COL,
    "cell_class_col": CELL_CLASS_COL,
    "pyg_obs_fields": PYG_EXTRA_OBS_FIELDS,

    "n_hvg": N_HVG,
    "n_hvg_lr": N_HVG_LR,
    "num_neighbors": NUM_NEIGHBORS,

    "per_file_hook": pancancer_per_file_hook,
    "apply_lr_corr_filter": APPLY_LR_CORR_FILTER,
    "skip_lr_filter_when_predefined": True,

    **DATA_REPRESENTATION_CONFIG,
}

lr_list_path_use = normalize_optional_path_local(LR_LIST_PATH)
gene_list_path_use = normalize_optional_path_local(GENE_LIST_PATH)

if lr_list_path_use is not None:
    base_config["lr_list_path"] = lr_list_path_use

if gene_list_path_use is not None:
    base_config["gene_list_path"] = gene_list_path_use

base_config

{'data_path_main': WindowsPath('D:/SpiderNet/Data/Pancancer'),
 'output_dir': WindowsPath('D:/SpiderNet/Results/Pancancer/ProcessedData_entire'),
 'adata_folder_name': 'adata_entire',
 'ligand_receptor_filedir_cellchatdb': WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_Cellchatdb.csv'),
 'ligand_receptor_filedir_scSeqComm': WindowsPath('D:/SpiderNet/SpiderNet_proj/SpiderNet_Project/SpiderNet/resources/Human_LR_pairs_scSeqComm.csv'),
 'sample_col': 'SampleID',
 'sample_name_obs_col': 'source_file_prefix',
 'sample_attr_obs_col': 'SampleID',
 'cell_class_col': 'celltype_final',
 'pyg_obs_fields': {},
 'n_hvg': 1000,
 'n_hvg_lr': 2000,
 'num_neighbors': 5,
 'per_file_hook': <function __main__.pancancer_per_file_hook(adata, context)>,
 'apply_lr_corr_filter': True,
 'skip_lr_filter_when_predefined': True,
 'expression_source': {'kind': 'X', 'name': None},
 'normalize_strategy': 'auto',
 'log1p': True,
 'remove_zero_count_cells': True,
 'spatia

## Step 4. Preview the LR-correlation density

Run this cell **before** the full loading step **only when no predefined LR list is supplied**.

If `LR_LIST_PATH` is provided, the original Pancancer loader skips LR activation/correlation
filtering, so this preview step is not needed.

In [5]:
# if APPLY_LR_CORR_FILTER:
#     preview = preview_lr_corr_distribution(base_config, show_plot=True)
# 
#     if preview.get("plot_path") is not None:
#         print("Preview plot saved to:", preview["plot_path"])
# else:
#     preview = None
#     print(
#         "A predefined LR list was provided. To match the original Pancancer loader, "
#         "LR activation/correlation filtering is skipped, so the preview step is not needed."
#     )

## Step 5. Set `lr_corr_threshold` and run the full loader

After inspecting the density plot above, set `LR_CORR_THRESHOLD` to the desired value.
In the original PerturbFISH loader, the LR-correlation filter uses `0.5`, so that is a
reasonable default starting point here.


In [6]:
LR_CORR_THRESHOLD = 0.2

In [7]:
config = dict(base_config)

if APPLY_LR_CORR_FILTER:
    if LR_CORR_THRESHOLD is None:
        raise ValueError(
            "LR_CORR_THRESHOLD cannot be None when LR correlation filtering is enabled. "
            "Please inspect the preview plot and set a threshold, for example 0.2."
        )
    config["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

bundle = prepare_processed_bundle_unified(config)

Step 1: Load the data and perform preprocessing
Reading AnnData files from: D:\SpiderNet\Data\Pancancer\adata_entire
HumanBreastCancerPatient1_subslice_0_annotated.h5ad: remove 65 zero-count cells
AnnData object with n_obs × n_vars = 47423 × 500
    obs: 'fov', 'volume', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'total_counts', 'celltype_final', 'subslice_id', 'CELL_TYPE', 'source_file_prefix', 'SampleID'
    obsm: 'spatial'
HumanBreastCancerPatient1_subslice_10_annotated.h5ad: remove 70 zero-count cells
HumanBreastCancerPatient1_subslice_11_annotated.h5ad: remove 189 zero-count cells
HumanBreastCancerPatient1_subslice_12_annotated.h5ad: remove 131 zero-count cells
HumanBreastCancerPatient1_subslice_13_annotated.h5ad: remove 196 zero-count cells
HumanBreastCancerPatient1_subslice_14_annotated.h5ad: remove 54 zero-count cells
HumanBreastCancerPatient1_subslice_15_annotated.h5ad: remove 118 zero-count cells
HumanBreastCancerPatient1_subslice_16_annotated.h5ad: remove 14

D:\SpiderNet\SpiderNet_proj\SpiderNet_Project\SpiderNet\dataloading_unified.py:1101: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc.AnnData.concatenate(*adata_file_list, batch_key="_concat_batch", index_unique=None)


Number of ligand-receptor pairs from CellChatDB: 210
Total number of Ligand–Receptor pairs after direct CellChat selection and branch-local deduplication: 149
Step 2: Build sample-level SpiderNet data dictionaries
Step 3: Build spatial neighbor graph
Total number of edges across all samples: 23538755
Step 4: One-hot encoding of cell types and PyG object creation
Step 5: Compute cell–cell–LRpair tensor
Step 6: Filter LR pairs based on coverage across edges
Quantiles of LR pair activation proportion: [0.00252275 0.01533454 0.06646801 0.16383648 0.80739075]
Step 7: Compute pairwise LR correlation across samples
Step 8: Finalize PyG features and cell labels
Step 9: Compute neighboring cell-type proportions
Step 11: Build batch-aligned adata_list
Step 12: Save processed objects


C:\Users\junji\miniconda3\envs\SpiderNet_env\Lib\site-packages\anndata\_io\utils.py:272: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Number of PyG samples: 160
Total tensor size GB: 19.48442817106843
Largest sample GB: 0.24842718243598938
Per-sample GB: [0.1963, 0.1681, 0.1259, 0.1819, 0.1269, 0.1633, 0.1314, 0.1503, 0.1391, 0.131, 0.1416, 0.1224, 0.1759, 0.1317, 0.1537, 0.1293, 0.1761, 0.1269, 0.1356, 0.1316, 0.1867, 0.1427, 0.1238, 0.1669, 0.1506, 0.1758, 0.1744, 0.1412, 0.1618, 0.1437, 0.1751, 0.1784, 0.221, 0.1568, 0.1636, 0.2005, 0.158, 0.214, 0.1409, 0.1862, 0.1354, 0.1095, 0.1267, 0.0937, 0.1338, 0.1294, 0.1542, 0.1081, 0.1421, 0.1266, 0.1123, 0.121, 0.1268, 0.1326, 0.0951, 0.1281, 0.1162, 0.1055, 0.1272, 0.1378, 0.068, 0.0737, 0.0709, 0.0953, 0.0914, 0.0583, 0.0795, 0.0655, 0.0618, 0.0861, 0.0744, 0.0762, 0.0748, 0.0626, 0.0543, 0.0847, 0.0728, 0.0761, 0.0638, 0.073, 0.1076, 0.108, 0.1391, 0.101, 0.1076, 0.1238, 0.1103, 0.0671, 0.0601, 0.1059, 0.1164, 0.0474, 0.0986, 0.1241, 0.0576, 0.1365, 0.113, 0.0454, 0.1065, 0.0516, 0.0551, 0.0617, 0.0386, 0.0235, 0.0322, 0.0477, 0.052, 0.0412, 0.0544, 0.0493, 0.0391, 0

## Step 6. Inspect the processed outputs

In [8]:
print("Output dir:", bundle["output_dir"])
print("Number of batches:", len(bundle["batch_cell_unique"]))
print("Total cells:", bundle["adata"].n_obs)
print("All genes retained before training subset:", len(bundle["genenames"]))
print("Training genes:", len(bundle["genenames_train"]))
print("Retained LR pairs:", len(bundle["LR_list"]))
print("Number of cell types:", bundle["adata"].obs[CELL_CLASS_COL].nunique())
print("First 10 batch labels:", bundle["batch_cell_unique"][:10])

bundle["adata"]

Output dir: D:\SpiderNet\Results\Pancancer\ProcessedData_entire
Number of batches: 160
Total cells: 4707751
All genes retained before training subset: 500
Training genes: 500
Retained LR pairs: 114
Number of cell types: 19
First 10 batch labels: ['HumanBreastCancerPatient1_subslice0'
 'HumanBreastCancerPatient1_subslice1'
 'HumanBreastCancerPatient1_subslice10'
 'HumanBreastCancerPatient1_subslice11'
 'HumanBreastCancerPatient1_subslice12'
 'HumanBreastCancerPatient1_subslice13'
 'HumanBreastCancerPatient1_subslice14'
 'HumanBreastCancerPatient1_subslice15'
 'HumanBreastCancerPatient1_subslice16'
 'HumanBreastCancerPatient1_subslice17']


AnnData object with n_obs × n_vars = 4707751 × 500
    obs: 'fov', 'volume', 'center_x', 'center_y', 'min_x', 'max_x', 'min_y', 'max_y', 'total_counts', 'celltype_final', 'subslice_id', 'CELL_TYPE', 'source_file_prefix', 'SampleID', '_concat_batch', 'B cell_prop', 'Breast-cancercell_prop', 'CD4_T_prop', 'CD8_T/NK_prop', 'Colon-cancercell_prop', 'DC_prop', 'Endothelial_prop', 'Epithelial_prop', 'Fibroblast_prop', 'Liver-cancercell_prop', 'Lung-cancercell_prop', 'Macrophage_prop', 'Mast cell_prop', 'Melanoma-cancercell_prop', 'Ovarian-cancercell_prop', 'Prostate-cancercell_prop', 'Treg_prop', 'Uterine-cancercell_prop', 'low_exp_prop'
    obsm: 'spatial'

## Step 7. Save Pancancer-specific batch metadata

This section writes one row per `SampleID` batch and saves it as
`metadata_sample.csv` in the processed output directory.

In [9]:
obs = bundle["adata"].obs.copy()

metadata_sample = (
    obs.groupby(SAMPLE_COL, dropna=False)
      .agg(
          source_file_prefix=("source_file_prefix", "first"),
          subslice_id=("subslice_id", "first"),
          num_cells=(SAMPLE_COL, "size"),
          n_cell_types=(CELL_CLASS_COL, pd.Series.nunique),
      )
      .reset_index()
)

celltype_counts = pd.crosstab(obs[SAMPLE_COL], obs[CELL_CLASS_COL]).reset_index()
celltype_counts.columns = [SAMPLE_COL] + [f"n_{col}" for col in celltype_counts.columns[1:]]

metadata_sample = metadata_sample.merge(celltype_counts, on=SAMPLE_COL, how="left")
metadata_sample = metadata_sample.sort_values([ "source_file_prefix", "subslice_id", SAMPLE_COL ]).reset_index(drop=True)

metadata_sample_path = bundle["output_dir"] / "metadata_sample.csv"
metadata_sample.to_csv(metadata_sample_path, index=False)

print("Saved:", metadata_sample_path)
display(metadata_sample.head())

C:\Users\junji\AppData\Local\Temp\ipykernel_21584\147077336.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby(SAMPLE_COL, dropna=False)


Saved: D:\SpiderNet\Results\Pancancer\ProcessedData_entire\metadata_sample.csv


,SampleID,source_file_prefix,subslice_id,num_cells,n_cell_types,n_B cell,n_Breast-cancercell,n_CD4_T,n_CD8_T/NK,n_Colon-cancercell,...,n_Liver-cancercell,n_Lung-cancercell,n_Macrophage,n_Mast cell,n_Melanoma-cancercell,n_Ovarian-cancercell,n_Prostate-cancercell,n_Treg,n_Uterine-cancercell,n_low_exp
0,HumanBreastCancerPatient1_subslice0,HumanBreastCancerPatient1,0,47423,11,236,26090,619,1184,0,...,0,0,3985,24,0,0,0,197,0,8
1,HumanBreastCancerPatient1_subslice1,HumanBreastCancerPatient1,1,40606,11,724,23753,579,1207,0,...,0,0,3204,27,0,0,0,212,0,4
2,HumanBreastCancerPatient1_subslice2,HumanBreastCancerPatient1,2,42502,11,256,18110,839,1517,0,...,0,0,4835,38,0,0,0,206,0,3
3,HumanBreastCancerPatient1_subslice3,HumanBreastCancerPatient1,3,31827,11,396,14044,856,1155,0,...,0,0,3419,22,0,0,0,167,0,1
4,HumanBreastCancerPatient1_subslice4,HumanBreastCancerPatient1,4,37125,11,198,20042,447,872,0,...,0,0,3226,52,0,0,0,177,0,6


## Step 8. Run MI dimension selection on the processed outputs

This section reuses the processed files that were just written to `bundle["output_dir"]`.
It does **not** rerun data loading or preprocessing.

By default, the MI-dimension-selection thresholds are chosen automatically based on the
number of retained LR pairs. You can leave the optional overrides below as `None` unless
you want to tune the heuristic manually.

In [11]:
from IPython.display import display
import pandas as pd

from SpiderNet.io import load_processed_data, get_spidernet_pyg_list_path, load_spidernet_pyg_list, spidernet_pyg_list_exists
from SpiderNet.MI_dimension_selection import run_mi_dimension_selection

# Optional advanced overrides for MI dimension selection.
# Leave these as None to use the default automatic heuristic.
MI_DIM_LR_SPEARCOR_THRESHOLD = None
MI_DIM_MIN_CLIQUE_SIZE = None
MI_DIM_JACCARD_THRESHOLD = None
MI_DIM_SHOW_HEATMAP = True

processed = load_processed_data(bundle["output_dir"])

print("Processed directory:", bundle["output_dir"])
print("Number of batches:", len(processed.spidernet_data))
print("Number of retained LR pairs:", len(processed.lr_list))
print("Number of training genes:", len(processed.genenames_train))

mi_dim_results = run_mi_dimension_selection(
    processed=processed,
    lr_list=processed.lr_list,
    output_dir=bundle["output_dir"],
    lr_spearcor_threshold=MI_DIM_LR_SPEARCOR_THRESHOLD,
    min_clique_size=MI_DIM_MIN_CLIQUE_SIZE,
    jaccard_thr=MI_DIM_JACCARD_THRESHOLD,
    show=MI_DIM_SHOW_HEATMAP,
)

mi_dim_summary_df = pd.DataFrame(
    [
        {
            "recommended_dim_envir": mi_dim_results["recommended_dim_envir"],
            "num_merged_subsets": mi_dim_results["num_merged_subsets"],
            "subset_sizes": mi_dim_results["subset_sizes"],
            "num_lr_pairs": mi_dim_results["num_lr_pairs"],
            "num_graph_edges": mi_dim_results["num_graph_edges"],
            "num_maximal_cliques_filtered": mi_dim_results["num_maximal_cliques_filtered"],
            "effective_min_clique_size": mi_dim_results["effective_min_clique_size"],
            "num_unassigned_lr_pairs": mi_dim_results["num_unassigned_lr_pairs"],
            "lr_spearcor_threshold": mi_dim_results["lr_spearcor_threshold"],
            "jaccard_thr": mi_dim_results["jaccard_thr"],
            "warning": mi_dim_results["warning"],
        }
    ]
)

display(mi_dim_summary_df)

print(f"Recommended dim_envir: {mi_dim_results['recommended_dim_envir']}")
print(f"Heatmap saved to: {mi_dim_results['heatmap_path']}")
print(f"Summary JSON saved to: {mi_dim_results['summary_path']}")
print(f"Subset summary CSV saved to: {mi_dim_results['subset_summary_path']}")

KeyboardInterrupt: 